# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [115]:
# Write your code below.

from dotenv import load_dotenv
load_dotenv()


True

In [116]:
import dask.dataframe as dd

import os
from glob import glob

# Get the path from the environment variable
price_data_dir = os.getenv('PRICE_DATA')

# List all parquet files in the directory
parquet_files = glob(os.path.join(price_data_dir, '**/*.parquet'),recursive=True)

parquet_files[:3]  # Just to show sample paths


['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet']

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [117]:
import os
from glob import glob
from dotenv import load_dotenv

# Write your code below.

# Load environment variables from .env file
load_dotenv()

# Get the value of the environment variable 'PRICE_DATA'
price_data_dir = os.getenv('PRICE_DATA')

# Use glob to find all .parquet files in the PRICE_DATA directory
parquet_files = glob(os.path.join(price_data_dir, '**/*.parquet'),recursive=True)

# Display a few of the files to confirm
parquet_files[:5]




['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet']

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [118]:

print("Parquet files found:", parquet_files)


Parquet files found: ['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\ACN\\AC

In [119]:
# Write your code below.
import dask.dataframe as dd

# Read all parquet files into a single Dask DataFrame
ddf = dd.read_parquet(parquet_files)

# Ensure data is sorted by Ticker and Date for correct lag and return computation
ddf.sort_values(by=['ticker', 'Date'])


ddf_2 = ddf.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1)
                        ,Close_Adj_lag_1 = x['Adj Close'].shift(1)            
                        ,returns=x['Close'] / x['Close'].shift(1)
                        ,high_low=x['High'] - x['Low']
                       )
)

C:\Users\vidhi\AppData\Local\Temp\ipykernel_20256\1237459349.py:11: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  ddf_2 = ddf.groupby('ticker', group_keys=False).apply(


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [120]:
# Write your code below.

pdf = ddf_2.compute()

pdf.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Roll_Avg_10 = x['returns'].rolling(10).mean()))



C:\Users\vidhi\AppData\Local\Temp\ipykernel_20256\3608254119.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pdf.groupby('ticker', group_keys=False).apply(


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,Close_Adj_lag_1,returns,high_low,Roll_Avg_10
218452,2020-02-19,16.709999,16.860001,16.702999,16.840000,16.609491,56500.0,RIV.csv,RIV,2020,NaN,NaN,NaN,0.157001,NaN
218453,2020-02-20,16.820000,16.920000,16.820000,16.920000,16.688396,48500.0,RIV.csv,RIV,2020,16.840000,16.609491,1.004751,0.100000,NaN
218454,2020-02-21,16.900000,16.950001,16.900000,16.940001,16.708122,52600.0,RIV.csv,RIV,2020,16.920000,16.688396,1.001182,0.050001,NaN
218455,2020-02-24,16.750000,16.750000,16.469999,16.549999,16.323460,132700.0,RIV.csv,RIV,2020,16.940001,16.708122,0.976977,0.280001,NaN
218456,2020-02-25,16.580000,16.618999,16.160000,16.219999,15.997976,118600.0,RIV.csv,RIV,2020,16.549999,16.323460,0.980060,0.459000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23026,2018-06-26,8.350000,8.530000,8.150000,8.475000,8.475000,97900.0,ALDX.csv,ALDX,2018,8.200000,8.200000,1.033537,0.380000,0.998780
23027,2018-06-27,8.500000,8.500000,7.900000,8.050000,8.050000,81900.0,ALDX.csv,ALDX,2018,8.475000,8.475000,0.949852,0.600000,0.994928
23028,2018-06-28,8.100000,8.200000,7.800000,8.150000,8.150000,61200.0,ALDX.csv,ALDX,2018,8.050000,8.050000,1.012422,0.400000,0.994405
23029,2018-06-29,8.100000,8.257000,7.800000,7.950000,7.950000,78000.0,ALDX.csv,ALDX,2018,8.150000,8.150000,0.975460,0.457000,0.989639


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

If data fits in memory, Pandas is often the most straightforward and effective choice.

It would have been better to use Dask only if the dataset was large or didn’t fit in memory. Dask handles big data efficiently through parallel and out-of-core processing. However, for small to medium datasets, pandas is faster and simpler, so using Dask would add unnecessary complexity.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [121]:
# Write your code below.



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.